# Section 1 : Environment Setup & Imports
Installing the required dependencies and load API keys.

## 1.1 installing dependencies

In [1]:
import warnings
warnings.filterwarnings('ignore')

# !pip install -r requirements.txt

## 1.2 Loading the OpenAI API key

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

# Section 2 : Data Chunking
loading the mock data and split it into chunks.

In [3]:
from rag_utils import DataProcessor

# Initialize the Data processor
processor = DataProcessor()

# Load and split the document
chunks = processor.load_and_split(directory_path="data/", chunk_size=300, chunk_overlap=50)

print(f"Total chunks created: {len(chunks)}")

sources = set([chunk.metadata.get("source") for chunk in chunks])
print(f"Sources loaded: {sources}")

100%|██████████| 1/1 [00:00<00:00, 275.85it/s]

Total chunks created: 170
Sources loaded: {'data/certifications.txt', 'data/Mohammad_CV.pdf', 'data/linkedin_profile.txt', 'data/github_projects.md'}


# Section 3 : Vector Database & Retriever
Convert the text chunks into embeddings using HuggingFace's `all-MiniLM-L6-v2` model and store them in a ChromaDB vector store.

In [4]:
# Create vector store
vectorstore = processor.create_vector_store(chunks, persist_directory="./chroma_db")
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Section 4 : Building the RAG Pipeline
inject our retriever into the `RAGPipeline` class. This class handles the prompt engineering and the LangChain Expression Language (LCEL) to ensure the LLM answers based *only* on the retrieved context and cites its sources.

In [5]:
from rag_chain import RAGPipeline

# Initialize the RAG pipeline with the retriever
rag_pipeline = RAGPipeline(retriever=retriever)
rag_chain = rag_pipeline.build()

# Section 5 : Inference

In [6]:
def query_rag(question: str):
    print(f"Question: {question}\n")

    # Invoke the chain
    result = rag_chain.invoke(question)

    print(f"Answer:\n{result['answer']}\n")

    print("Source Documents Used:")
    for i, doc in enumerate(result["context"]):
        # Print the source file name along with the content
        source_name = doc.metadata.get("source", "Unknown")
        print(f"  [Source {i+1} - {source_name}]:")
        print(f"  {doc.page_content.strip()}\n")
    print("-" * 50)

## Test 1: Specific skill question (Tests Skills & Languages section)

In [7]:
query_rag("What are Mohammad's main programming languages?")

Question: What are Mohammad's main programming languages?

Answer:
Mohammad's main programming languages are Python, Swift, C++, and Dart (used with Flutter). He also works with iOS development, which primarily uses Swift.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  LANGUAGES
Arabic is Mohammad's native language. He also works with English in technical, academic, and professional environments.

  [Source 2 - data/linkedin_profile.txt]:
  Mohammad's technical skills include Python, Swift, C++, Dart, Flutter, TensorFlow, TensorFlow Lite, machine learning, computer vision, natural language processing, retrieval-augmented generation, embeddings, vector databases, Firebase, REST APIs, SQLite, Git, GitHub, iOS development, software

  [Source 3 - data/certifications.txt]:
  Mohammad has continued professional development in object-oriented programming, software architecture, networking, databases, API integration, version control, testing concepts, debugging, and appl

## Test 2: Experience question (Tests Professional Experience section)

In [8]:
query_rag("Where did Mohammad work and what did he do there?")

Question: Where did Mohammad work and what did he do there?

Answer:
Mohammad works as an AI Engineer at Cellula, where he tackles practical artificial intelligence problems using real data. His responsibilities include understanding AI problems, preparing and examining data, selecting suitable machine‑learning approaches, experimenting with models, and evaluating their performance.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Mohammad's technical skills include Python, Swift, C++, Dart, Flutter, TensorFlow, TensorFlow Lite, machine learning, computer vision, natural language processing, retrieval-augmented generation, embeddings, vector databases, Firebase, REST APIs, SQLite, Git, GitHub, iOS development, software

  [Source 2 - data/linkedin_profile.txt]:
  LANGUAGES
Arabic is Mohammad's native language. He also works with English in technical, academic, and professional environments.

  [Source 3 - data/certifications.txt]:
  As part of his RAG experiments, Mo

## Test 3: GitHub question (Tests Projects section)

In [9]:
query_rag("What kind of projects has Mohammad worked on?")

Question: What kind of projects has Mohammad worked on?

Answer:
Mohammad has worked on a variety of AI‑focused projects, including machine‑learning and computer‑vision systems, retrieval‑augmented generation (RAG) solutions, mobile AI applications, and agricultural‑technology initiatives. As an AI Engineer at Cellula, he tackles practical AI problems using real data—understanding the problem, preparing and examining data, selecting appropriate ML approaches, experimenting with models, and evaluating results. His work also spans software engineering practices, such as building maintainable, understandable applications using Python, Swift, C++, Dart, Flutter, TensorFlow, TensorFlow Lite, Firebase, REST APIs, SQLite, and version‑control tools like Git and GitHub.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Mohammad is interested in AI engineering, machine learning, computer vision, RAG systems, mobile AI, agricultural technology, software engineering, and practica

## Test 4: Certifications question (Tests Certifications section)

In [10]:
query_rag("What certifications does Mohammad have?")

Question: What certifications does Mohammad have?

Answer:
I don't have enough information to answer that.

Source Documents Used:
  [Source 1 - data/certifications.txt]:
  Mohammad approaches certifications and professional development as part of continuous learning rather than as isolated achievements. He regularly studies new AI architectures, retrieval techniques, software engineering practices, mobile development technologies, and algorithmic problem solving. His

  [Source 2 - data/linkedin_profile.txt]:
  Mohammad's technical skills include Python, Swift, C++, Dart, Flutter, TensorFlow, TensorFlow Lite, machine learning, computer vision, natural language processing, retrieval-augmented generation, embeddings, vector databases, Firebase, REST APIs, SQLite, Git, GitHub, iOS development, software

  [Source 3 - data/linkedin_profile.txt]:
  Mohammad is interested in AI engineering, machine learning, computer vision, RAG systems, mobile AI, agricultural technology, software engineer

## Test 5: Out-of-scope question (testing hallucination guardrails)

In [11]:
query_rag("What is Mohammad's favorite food?")

Question: What is Mohammad's favorite food?

Answer:
I don't have enough information to answer that.

Source Documents Used:
  [Source 1 - data/linkedin_profile.txt]:
  Mohammad's technical skills include Python, Swift, C++, Dart, Flutter, TensorFlow, TensorFlow Lite, machine learning, computer vision, natural language processing, retrieval-augmented generation, embeddings, vector databases, Firebase, REST APIs, SQLite, Git, GitHub, iOS development, software

  [Source 2 - data/linkedin_profile.txt]:
  Mohammad is interested in AI engineering, machine learning, computer vision, RAG systems, mobile AI, agricultural technology, software engineering, and practical applications of artificial intelligence. His long-term goal is to contribute to technology projects that solve meaningful problems in the

  [Source 3 - data/linkedin_profile.txt]:
  LANGUAGES
Arabic is Mohammad's native language. He also works with English in technical, academic, and professional environments.

  [Source 4 - da